<a href="https://colab.research.google.com/github/sasirekha52/Antartic-Anlytics-Engine/blob/main/Antartic_anlytics_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install polars pyarrow streamlit plotly pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 100.3 MB/s eta 0:00:00


In [ ]:
import requests

url = "https://api.gbif.org/v1/occurrence/download/request/0042620-260519110011954.zip"

response = requests.get(url)

with open("antarctica.zip", "wb") as f:
    f.write(response.content)

print("Download Complete")

Download Complete


In [ ]:
import zipfile

with zipfile.ZipFile("antarctica.zip", "r") as z:
    z.extractall("gbif")

print("Extraction Complete")

Extraction Complete


In [ ]:
import os

for file in os.listdir("gbif"):
    print(file)

occurrence.txt
citations.txt
metadata.xml
rights.txt
meta.xml
verbatim.txt
dataset
multimedia.txt


In [ ]:
import polars as pl

lf = pl.scan_csv(
    "gbif/occurrence.txt",
    separator="\t",
    ignore_errors=True,
    infer_schema_length=10000,
    quote_char=None,
    null_values=['"In Alcohol"; Entered from ledger."']
)

print(lf)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Csv SCAN [gbif/occurrence.txt]
PROJECT */230 COLUMNS
ESTIMATED ROWS: 9294789


In [ ]:
lf.sink_parquet(
    "antarctica.parquet"
)

print("Parquet Created")

Parquet Created


In [ ]:
lf = pl.scan_parquet(
    "antarctica.parquet"
)

print(lf)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

Parquet SCAN [antarctica.parquet]
PROJECT */230 COLUMNS
ESTIMATED ROWS: 10356823


In [ ]:
total = lf.select(
    pl.len()
).collect()

print(total)

shape: (1, 1)
┌──────────┐
│ len      │
│ ---      │
│ u32      │
╞══════════╡
│ 10356823 │
└──────────┘


In [ ]:
species_count = (
    lf.select(
        pl.col("species")
        .n_unique()
    )
    .collect()
)

species_count

species
u32
33920


In [ ]:
top_species = (
    lf.group_by("species")
      .len()
      .sort("len", descending=True)
      .limit(20)
      .collect()
)

top_species

species,len
str,u32
null,4641572
"""Mirounga leonina""",1077727
"""Leptonychotes weddellii""",406112
"""Arctocephalus gazella""",226954
"""Pygoscelis adeliae""",196543
…,…
"""Procellaria aequinoctialis""",53607
"""Phoebetria palpebrata""",46867
"""Thalassoica antarctica""",46033


In [ ]:
timeline = (
    lf.group_by("year")
      .len()
      .sort("year")
      .collect()
)

timeline

year,len
i64,u32
null,313998
1668,1
1697,1
1776,1
1820,3
…,…
2022,140323
2023,95243
2024,134146


In [ ]:
#Species Explorer

import polars as pl

species_explorer = (
    lf.select([
        "species",
        "genus",
        "family",
        "order",
        "class",
        "phylum",
        "kingdom"
    ])
    .unique()
    .collect()
)

species_explorer.head()

species,genus,family,order,class,phylum,kingdom
str,str,str,str,str,str,str
"""Onykia robusta""","""Onykia""","""Onychoteuthidae""","""Oegopsida""","""Cephalopoda""","""Mollusca""","""Animalia"""
"""Forsterygion flavonigrum""","""Forsterygion""","""Tripterygiidae""","""Perciformes""",null,"""Chordata""","""Animalia"""
"""Colibri cyanotus""","""Colibri""","""Trochilidae""","""Apodiformes""","""Aves""","""Chordata""","""Animalia"""
"""Hemigrapsus sexdentatus""","""Hemigrapsus""","""Varunidae""","""Decapoda""","""Malacostraca""","""Arthropoda""","""Animalia"""
null,"""Hemisphaerammina""","""Stegnamminidae""","""Astrorhizida""",null,"""Foraminifera""","""Chromista"""


In [ ]:
search_species = "Pygoscelis"

result = (
    species_explorer
    .filter(
        pl.col("species")
        .str.contains(
            search_species,
            literal=False
        )
    )
)

result

species,genus,family,order,class,phylum,kingdom
str,str,str,str,str,str,str
"""Pygoscelis papua""","""Pygoscelis""","""Spheniscidae""","""Sphenisciformes""","""Aves""","""Chordata""","""Animalia"""
"""Pygoscelis antarcticus""","""Pygoscelis""","""Spheniscidae""","""Sphenisciformes""","""Aves""","""Chordata""","""Animalia"""
"""Pygoscelis adeliae""","""Pygoscelis""","""Spheniscidae""","""Sphenisciformes""","""Aves""","""Chordata""","""Animalia"""


In [ ]:
#Taxonomy Browser

taxonomy = (
    lf.group_by([
        "kingdom",
        "phylum",
        "class"
    ])
    .len()
    .sort(
        "len",
        descending=True
    )
    .collect()
)

taxonomy.head(50)

kingdom,phylum,class,len
str,str,str,u32
"""Animalia""","""Chordata""","""Mammalia""",2024054
"""Animalia""","""Chordata""","""Aves""",1676850
"""Bacteria""","""Proteobacteria""","""Alphaproteobacteria""",491526
"""Animalia""","""Arthropoda""","""Copepoda""",465160
"""Animalia""","""Chordata""",null,424194
…,…,…,…
"""Animalia""","""Echinodermata""","""Ophiuroidea""",29074
"""Animalia""","""Mollusca""","""Bivalvia""",27597
"""Chromista""",null,null,26637


In [ ]:
taxonomy = (
    lf.group_by("kingdom")
      .len()
      .collect()
)

fig = px.pie(
    taxonomy.to_pandas(),
    names="kingdom",
    values="len",
    title="Kingdom Distribution"
)

fig.show()

In [ ]:
timeline = (
    lf.group_by("year")
      .len()
      .sort("year")
      .collect()
)

fig = px.line(
    timeline.to_pandas(),
    x="year",
    y="len",
    markers=True,
    title="Observation Timeline"
)

fig.show()

In [ ]:
import plotly.express as px

top_species = (
    lf.group_by("species")
      .len()
      .sort("len", descending=True)
      .limit(20)
      .collect()
)

fig = px.bar(
    top_species.to_pandas(),
    x="species",
    y="len",
    title="Top 20 Species"
)

fig.show()

In [ ]:
country = (
    lf.group_by("countryCode")
      .len()
      .sort("len", descending=True)
      .limit(20)
      .collect()
)

fig = px.bar(
    country.to_pandas(),
    x="countryCode",
    y="len",
    title="Country Distribution"
)

fig.show()

In [59]:
genus = (
    lf.group_by("genus")
      .len()
      .sort("len", descending=True)
      .limit(10)
      .collect()
)

fig = px.bar(
    genus.to_pandas(),
    x="genus",
    y="len",
    title="Top 10 Genera"
)

fig.show()

In [ ]:
rare = (
    lf.group_by("species")
      .len()
      .filter(pl.col("len") < 5)
      .collect()
)

rare.head(20)

species,len
str,u32
"""Ptilella inflata""",2
"""Leptogium antarcticum""",4
"""Navicula subpolaris""",1
"""Arbacia lixula""",1
"""Caloneis bacillum""",2
…,…
"""Tulostoma macrocephalum""",1
"""Lagena herdmani""",1
"""Tapinoma atriceps""",1


In [60]:
import polars as pl
import plotly.express as px

heatmap = (
    lf.drop_nulls([
        "decimalLatitude",
        "decimalLongitude"
    ])
    .group_by([
        "decimalLatitude",
        "decimalLongitude"
    ])
    .agg(
        pl.col("species")
        .n_unique()
        .alias("species_count")
    )
    .limit(5000)
    .collect()
)

fig = px.scatter_geo(
    heatmap.to_pandas(),
    lat="decimalLatitude",
    lon="decimalLongitude",
    size="species_count",
    color="species_count",
    title="Species Heatmap"
)

fig.show()

In [58]:
quality = (
    lf.select([
        pl.col("species").is_null().sum().alias("Missing Species"),
        pl.col("year").is_null().sum().alias("Missing Year"),
        pl.col("decimalLatitude").is_null().sum().alias("Missing Lat"),
        pl.col("decimalLongitude").is_null().sum().alias("Missing Lon")
    ])
    .collect()
)

quality

Missing Species,Missing Year,Missing Lat,Missing Lon
u32,u32,u32,u32
4641572,313998,0,0
